In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 61.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=d66334605e1e1b6c8c305e6065494b11bb3d939613fd4a83d3c786aaddae88be
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [3]:
# SETUP & QUANTUM RANDOM NUMBER GENERATOR
num_bits = 50 # Length of the initial sequence
simulator = BasicSimulator()

def generate_quantum_random_bits(length):
    """Generates random bits by measuring a qubit in superposition."""
    bits = []
    for _ in range(length):
        qc = QuantumCircuit(1, 1)
        qc.h(0) # Create superposition |+>
        qc.measure(0, 0)
        t_qc = transpile(qc, simulator)
        result = simulator.run(t_qc, shots=1).result()
        bit = int(list(result.get_counts().keys())[0])
        bits.append(bit)
    return bits

In [4]:
# ALICE'S ACTIONS
print("--- ALICE ---")
# Alice generates random bits for her message and her choice of bases
# 0 represents Rectilinear (Z) basis, 1 represents Diagonal (X) basis
alice_bits = generate_quantum_random_bits(num_bits)
alice_bases = generate_quantum_random_bits(num_bits)

print(f"Alice's bits:  {alice_bits}")
print(f"Alice's bases: {alice_bases}")

# Alice encodes her bits into quantum states
encoded_qubits = []
for i in range(num_bits):
    qc = QuantumCircuit(1, 1)
    # If bit is 1, apply X gate to flip to |1>
    if alice_bits[i] == 1:
        qc.x(0)
    # If basis is 1 (Diagonal), apply H gate to switch to |+> or |->
    if alice_bases[i] == 1:
        qc.h(0)
    encoded_qubits.append(qc)

--- ALICE ---
Alice's bits:  [1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
Alice's bases: [0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0]


In [5]:
# BOB'S ACTIONS
print("\n--- BOB ---")
# Bob generates his own random choices for measurement bases
bob_bases = generate_quantum_random_bits(num_bits)
bob_bits = []

print(f"Bob's bases:   {bob_bases}")

# Bob measures the qubits he received from Alice
for i in range(num_bits):
    qc = encoded_qubits[i]
    # If Bob chose the Diagonal basis, he applies an H gate before measuring
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)

    t_qc = transpile(qc, simulator)
    result = simulator.run(t_qc, shots=1).result()
    measured_bit = int(list(result.get_counts().keys())[0])
    bob_bits.append(measured_bit)

print(f"Bob's bits:    {bob_bits}")


--- BOB ---
Bob's bases:   [1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1]
Bob's bits:    [1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]


In [6]:
# PUBLIC COMMUNICATION & SIFTING
print("\n--- PUBLIC COMMUNICATION ---")
# Alice and Bob share their bases publicly and keep only the bits where they matched
sifted_key_alice = []
sifted_key_bob = []

for i in range(num_bits):
    if alice_bases[i] == bob_bases[i]:
        sifted_key_alice.append(alice_bits[i])
        sifted_key_bob.append(bob_bits[i])

print(f"Sifted Key Length: {len(sifted_key_alice)}")


--- PUBLIC COMMUNICATION ---
Sifted Key Length: 16


In [7]:
# ERROR CHECKING
# They sacrifice a portion of their sifted key to check for eavesdropping
sample_size = len(sifted_key_alice) // 2
sample_alice = sifted_key_alice[:sample_size]
sample_bob = sifted_key_bob[:sample_size]

errors = sum(1 for a, b in zip(sample_alice, sample_bob) if a != b)
error_rate = errors / sample_size if sample_size > 0 else 0

print(f"\nError check on {sample_size} bits.")
print(f"Errors found: {errors}")
print(f"Error Rate: {error_rate * 100:.2f}%")

if error_rate == 0:
    final_key = sifted_key_alice[sample_size:]
    print("\nSuccess! No eavesdropper detected.")
    print(f"Final Secret Key: {final_key}")
else:
    print("\nWarning: Errors detected. Protocol aborted.")


Error check on 8 bits.
Errors found: 0
Error Rate: 0.00%

Success! No eavesdropper detected.
Final Secret Key: [1, 0, 0, 1, 0, 1, 0, 0]
